In [1]:
from datetime import date, datetime

# ---------------- Prototype limits (easy to change) ----------------
LOW_AVAILABILITY_LIMIT = 85.0      # % - below this = low availability
EXCESSIVE_DOWNTIME_HOURS = 100.0   # hours - above this = excessive downtime
REPEATED_ALERT_LIMIT = 3           # alerts - this many or more = repeated alerts
DUE_SOON_DAYS = 14                 # days - maintenance due within this window = "due"

UNDER_MAINTENANCE_LABELS = {"under maintenance", "in maintenance", "maintenance"}


def calculate_availability(operating_time, downtime):
    """Availability (%) = operating / (operating + downtime) * 100."""
    try:
        operating_time = float(operating_time)
        downtime = float(downtime)
    except (TypeError, ValueError):
        raise ValueError("Operating time and downtime must be numbers.")
    if operating_time < 0 or downtime < 0:
        raise ValueError("Operating time and downtime cannot be negative.")
    total = operating_time + downtime
    if total == 0:
        raise ValueError("Operating time and downtime cannot both be zero.")
    return round(operating_time / total * 100, 2)


def _to_date(value):
    """Convert a string / datetime / date into a date object (None if empty)."""
    if value is None or value == "":
        return None
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    try:
        return datetime.strptime(str(value).strip()[:10], "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(f"Date {value!r} must be in YYYY-MM-DD format.")


def maintenance_due_status(next_maintenance_date, today=None):
    """Return 'OVERDUE', 'DUE', or 'OK' depending on the next maintenance date."""
    next_date = _to_date(next_maintenance_date)
    if next_date is None:
        return "UNKNOWN"
    today = _to_date(today) or date.today()
    days_left = (next_date - today).days
    if days_left < 0:
        return "OVERDUE"
    if days_left <= DUE_SOON_DAYS:
        return "DUE"
    return "OK"


def is_under_maintenance(maintenance_status):
    return str(maintenance_status).strip().lower() in UNDER_MAINTENANCE_LABELS


def assess_equipment(equipment_id, operating_time, downtime, maintenance_status,
                     next_maintenance_date, alert_count=0, today=None):
    """
    Assess ONE piece of equipment. Returns a dict of results and a list of flags.
    """
    availability = calculate_availability(operating_time, downtime)
    due_status = maintenance_due_status(next_maintenance_date, today)
    flags = []

    if is_under_maintenance(maintenance_status):
        flags.append("Under maintenance")
    if due_status == "OVERDUE":
        flags.append("Overdue for maintenance")
    elif due_status == "DUE":
        flags.append("Due for maintenance")
    if float(downtime) > EXCESSIVE_DOWNTIME_HOURS:
        flags.append("Excessive downtime")
    if availability < LOW_AVAILABILITY_LIMIT:
        flags.append("Low availability")
    if int(alert_count) >= REPEATED_ALERT_LIMIT:
        flags.append("Repeated alerts")

    return {
        "equipment_id": equipment_id,
        "availability": availability,
        "due_status": due_status,
        "flags": flags,
        "needs_attention": len(flags) > 0,
    }


def monitor_maintenance_dataframe(df, today=None):
    """
    Assess a whole pandas DataFrame.
    Needs columns: 'Equipment ID', 'Operating Hours', 'Downtime',
                   'Maintenance Status', 'Next Maintenance Date', 'Alert Count'
    Adds columns: Availability, Maintenance Due, Flags, and True/False columns
    for each category so the dashboard can filter easily.
    """
    out = df.copy()
    availability, due, flags_text = [], [], []
    for _, row in out.iterrows():
        try:
            r = assess_equipment(
                row["Equipment ID"], row["Operating Hours"], row["Downtime"],
                row["Maintenance Status"], row["Next Maintenance Date"],
                row.get("Alert Count", 0), today,
            )
            availability.append(r["availability"])
            due.append(r["due_status"])
            flags_text.append("; ".join(r["flags"]) if r["flags"] else "None")
        except ValueError:
            availability.append(None)
            due.append("INVALID")
            flags_text.append("Invalid data")
    out["Availability"] = availability
    out["Maintenance Due"] = due
    out["Flags"] = flags_text

    out["Under Maintenance"] = out["Flags"].str.contains("Under maintenance")
    out["Overdue"] = out["Flags"].str.contains("Overdue")
    out["Due"] = out["Flags"].str.contains("Due for")
    out["Excessive Downtime"] = out["Flags"].str.contains("Excessive downtime")
    out["Low Availability"] = out["Flags"].str.contains("Low availability")
    out["Repeated Alerts"] = out["Flags"].str.contains("Repeated alerts")
    return out


def maintenance_summary(assessed_df):
    """Numbers for the dashboard KPIs."""
    total = len(assessed_df)
    valid = assessed_df["Availability"].dropna()
    return {
        "total_equipment": total,
        "average_availability": round(float(valid.mean()), 2) if len(valid) else 0.0,
        "under_maintenance": int(assessed_df["Under Maintenance"].sum()),
        "due": int(assessed_df["Due"].sum()),
        "overdue": int(assessed_df["Overdue"].sum()),
        "excessive_downtime": int(assessed_df["Excessive Downtime"].sum()),
        "low_availability": int(assessed_df["Low Availability"].sum()),
        "repeated_alerts": int(assessed_df["Repeated Alerts"].sum()),
    }


def maintenance_priority(assessed_df):
    """Rank equipment: more flags and lower availability = higher priority."""
    ranked = assessed_df.copy()
    ranked["Flag Count"] = ranked["Flags"].apply(
        lambda f: 0 if f in ("None", "Invalid data") else len(f.split("; ")))
    return ranked.sort_values(["Flag Count", "Availability"],
                              ascending=[False, True]).reset_index(drop=True)


# ---------------------------------------------------------------
# Quick self-test: run `python maintenance.py`
# ---------------------------------------------------------------
if __name__ == "__main__":
    import pandas as pd

    sample = pd.DataFrame({
        "Equipment ID": ["TRK-001", "TRK-012", "LDR-004", "CNV-002"],
        "Operating Hours": [900, 600, 700, 950],
        "Downtime": [40, 180, 20, 30],
        "Maintenance Status": ["Operational", "Under Maintenance", "Operational", "Operational"],
        "Next Maintenance Date": ["2026-11-20", "2026-10-10", "2026-09-15", "2026-10-12"],
        "Alert Count": [0, 4, 1, 3],
    })
    result = monitor_maintenance_dataframe(sample, today="2026-10-03")
    print(result[["Equipment ID", "Availability", "Maintenance Due", "Flags"]].to_string(index=False))
    print()
    print(maintenance_summary(result))
    print()
    print(maintenance_priority(result)[["Equipment ID", "Flag Count", "Availability"]])


Equipment ID  Availability Maintenance Due                                                                                         Flags
     TRK-001         95.74              OK                                                                                          None
     TRK-012         76.92             DUE Under maintenance; Due for maintenance; Excessive downtime; Low availability; Repeated alerts
     LDR-004         97.22         OVERDUE                                                                       Overdue for maintenance
     CNV-002         96.94             DUE                                                          Due for maintenance; Repeated alerts

{'total_equipment': 4, 'average_availability': 91.7, 'under_maintenance': 1, 'due': 2, 'overdue': 1, 'excessive_downtime': 1, 'low_availability': 1, 'repeated_alerts': 2}

  Equipment ID  Flag Count  Availability
0      TRK-012           5         76.92
1      CNV-002           2         96.94
2      LDR-004     